In [2]:
!pip install SimpleITK

In [ ]:
import os
import glob
import numpy as np
import cv2
import pandas as pd
import SimpleITK as sitk
from tqdm import tqdm

# --- 1. Define Exact Kaggle Paths ---


COVID_BASE = '/kaggle/input/datasets/nguyentienda32143/covid-19-ct-lung-and-infection-segmentation/data/data'
COVID_IMG_DIR = os.path.join(COVID_BASE, 'Covid-19-2')
COVID_INF_MASK_DIR = os.path.join(COVID_BASE, 'InfectionMasks')
COVID_LUNG_MASK_DIR = os.path.join(COVID_BASE,'LungMasks')

LUNA_BASE = '/kaggle/input/datasets/fanbyprinciple/luna-lung-cancer-dataset'
LUNA_IMG_DIR = os.path.join(LUNA_BASE, 'seg-lungs-LUNA16/seg-lungs-LUNA16')
LUNA_ANNOTATIONS = os.path.join(LUNA_BASE, 'annotations.csv')

# --- Output Directories ---
OUTPUT_IMG_DIR = '/kaggle/working/processed_data/images'
OUTPUT_MASK_DIR = '/kaggle/working/processed_data/masks'
os.makedirs(OUTPUT_IMG_DIR, exist_ok=True)
os.makedirs(OUTPUT_MASK_DIR, exist_ok=True)

In [9]:

def window_image(image, wl=-600, ww=1500):
    """Applies CT Lung Windowing"""
    hu_min = wl - (ww / 2)
    hu_max = wl + (ww / 2)
    image = np.clip(image, hu_min, hu_max)
    return (image - hu_min) / (hu_max - hu_min)

# --- 2. Process COVID-19 Data (Already 2D Slices) ---
def process_covid_slices():
    print("Processing COVID-19 2D Slices...")
    # Get all infection mask files
    inf_masks = sorted(glob.glob(os.path.join(COVID_INF_MASK_DIR, '*.*')))
    
    for inf_path in tqdm(inf_masks[:500]): # Test with 500 slices first to save time
        filename = os.path.basename(inf_path)
        img_path = os.path.join(COVID_IMG_DIR, filename)
    
        if os.path.exists(img_path):
            # Load images (using cv2 assuming they are png/jpg/tif. If .npy, use np.load)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            inf_mask = cv2.imread(inf_path, cv2.IMREAD_GRAYSCALE)
            
            if img is not None and inf_mask is not None:
                # Resize to our target 256x256
                img_resized = cv2.resize(img, (256, 256), interpolation=cv2.INTER_LINEAR)
                inf_mask_resized = cv2.resize(inf_mask, (256, 256), interpolation=cv2.INTER_NEAREST)
                
                save_name = f"covid_{filename.split('.')[0]}.npy"
                np.save(os.path.join(OUTPUT_IMG_DIR, save_name), img_resized.astype(np.float32))
                np.save(os.path.join(OUTPUT_MASK_DIR, save_name), inf_mask_resized.astype(np.uint8))    


process_covid_slices()

Processing COVID-19 2D Slices...


100%|██████████| 500/500 [00:10<00:00, 46.55it/s]


Processing LUNA16 .mhd volumes...


100%|██████████| 2/2 [00:01<00:00,  1.91it/s]

Data extraction test complete! Check /kaggle/working/processed_data/


In [11]:
def world_to_voxel(world_coord, origin, spacing):
    """Translates from physical mm to pixel indices using the formula."""
    stretched_voxel_coord = np.absolute(world_coord - origin)
    voxel_coord = stretched_voxel_coord / spacing
    return np.round(voxel_coord).astype(int)

def process_luna16_mhd_with_masks():
    print("Processing LUNA16 .mhd volumes and generating nodule masks...")
    mhd_files = sorted(glob.glob(os.path.join(LUNA_IMG_DIR, '*.mhd')))
    
    try:
        annotations = pd.read_csv(LUNA_ANNOTATIONS)
    except FileNotFoundError:
        print(f"Error: {LUNA_ANNOTATIONS} not found.")
        return

    
    for mhd_path in tqdm(mhd_files[:]): 
        itk_img = sitk.ReadImage(mhd_path)
        img_array = sitk.GetArrayFromImage(itk_img) # (z, y, x)
        
        # 1. Get Physical Metadata
        origin = np.array(itk_img.GetOrigin())      # (x, y, z)
        spacing = np.array(itk_img.GetSpacing())    # (x, y, z)
        
        patient_id = os.path.basename(mhd_path).replace('.mhd', '')
        
        # 2. Find nodules for this specific patient
        patient_nodules = annotations[annotations['seriesuid'] == patient_id]
        
        # 3. Create a blank 3D mask
        mask_array = np.zeros_like(img_array, dtype=np.uint8)
        
        # 4. Draw the nodules onto the 3D mask
        for _, nodule in patient_nodules.iterrows():
            world_coord = np.array([nodule['coordX'], nodule['coordY'], nodule['coordZ']])
            diameter_mm = nodule['diameter_mm']
            
            voxel_coord = world_to_voxel(world_coord, origin, spacing)
            
            # The array is (z,y,x) but the coordinates are (x,y,z)
            v_x, v_y, v_z = voxel_coord
            radius_px = int((diameter_mm / 2) / spacing[0]) # Approximate radius in pixels
            
            # Draw a simple sphere in the 3D numpy array
            z_min, z_max = max(0, v_z - radius_px), min(mask_array.shape[0], v_z + radius_px + 1)
            y_min, y_max = max(0, v_y - radius_px), min(mask_array.shape[1], v_y + radius_px + 1)
            x_min, x_max = max(0, v_x - radius_px), min(mask_array.shape[2], v_x + radius_px + 1)
            
            for z in range(z_min, z_max):
                for y in range(y_min, y_max):
                    for x in range(x_min, x_max):
                        if (z - v_z)**2 + (y - v_y)**2 + (x - v_x)**2 <= radius_px**2:
                            mask_array[z, y, x] = 1 # Mark as Nodule (Class 1 for LUNA)
        
        # 5. Extract ONLY the 2D slices that contain nodules (to keep data balanced)
        img_array_windowed = window_image(img_array)
        
        for z_slice in range(img_array.shape[0]):
            if np.max(mask_array[z_slice, :, :]) > 0: # If there is a nodule in this slice
                img_slice = cv2.resize(img_array_windowed[z_slice, :, :], (256, 256), interpolation=cv2.INTER_LINEAR)
                mask_slice = cv2.resize(mask_array[z_slice, :, :], (256, 256), interpolation=cv2.INTER_NEAREST)
                
                save_name = f"luna_{patient_id}_slice_{z_slice}.npy"
                np.save(os.path.join(OUTPUT_IMG_DIR, save_name), img_slice.astype(np.float32))
                np.save(os.path.join(OUTPUT_MASK_DIR, save_name), mask_slice.astype(np.uint8))

# Call the new function
process_luna16_mhd_with_masks()


Processing LUNA16 .mhd volumes and generating nodule masks...


100%|██████████| 888/888 [06:01<00:00,  2.46it/s]
